# Transformer Lab — Tiny Autoregressive LM (Offline) — AI Lab Instructor Notebook

*NLP & Transformers · Hard*

Build a tiny character-level autoregressive language model in NumPy using single-head causal self-attention, next-token cross-entropy, manual backpropagation, SGD training, and sampling.

**Outcome.** By the end, you will implement teacher forcing, causal masking, a minimal attention-based LM forward/backward pass, train it end to end, and sample text autoregressively.

9 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import numpy as np

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)


## Task 1: Dataset & Setup

# Tiny Autoregressive LM (NumPy) — Student Lab

We train a tiny **character-level autoregressive language model** with **single-head causal self-attention**.

What you implement in this lab:
- vocabulary + encoding/decoding
- next-token batching with teacher forcing
- stable softmax + causal masking
- attention forward pass
- cross-entropy loss
- manual backpropagation
- SGD training
- text sampling

This is an offline NumPy lab. The point is to understand the mechanics of a tiny decoder-style LM end to end.

In [ ]:
import numpy as np

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

**Why this works.** # Tiny Autoregressive LM (NumPy) — Instructor Lab

This setup cell only defines the shared NumPy environment, deterministic RNG, and a small `check(...)` helper used by the later tasks.

## Task 2: Section 0 — Corpus + vocab
*Section: Corpus + vocab*

## Section 0 — Corpus + vocab

Build a **character vocabulary** and the two conversion functions:
- `encode(text) -> ids`
- `decode(ids) -> text`

Why char-level here?
- it keeps the lab fully offline
- it makes every token and transition easy to inspect
- it avoids hiding complexity inside a tokenizer library

In [ ]:
corpus = """the quick brown fox jumps over the lazy dog
pack my box with five dozen liquor jugs
sphinx of black quartz, judge my vow
how vexingly quick daft zebras jump
"""
chars = sorted(list(set(corpus)))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
V = len(chars)

def encode(s: str) -> np.ndarray:
    return np.array([stoi[c] for c in s], dtype=np.int64)

def decode(ids: np.ndarray) -> str:
    return ''.join(itos[int(i)] for i in ids)

data = encode(corpus)
print('vocab', V, 'len', data.size)
print('snippet:', decode(data[:80]))


In [ ]:
# Checks
check('roundtrip', decode(encode('the quick')) == 'the quick')
check('data_dtype', data.dtype == np.int64)
check('vocab_positive', V > 10 and data.size == len(corpus))


**Why this works.** Character-level encoding keeps the lab interpretable. `stoi` maps characters to integer ids, `itos` reverses that mapping, and `encode` / `decode` let the rest of the model work with integer sequences.

## Task 3: Section 1 — Next-token batches
*Section: Next-token batches*

## Section 1 — Next-token batches

Create training pairs using **teacher forcing**.

For a sequence length `T`:
- input `X` contains tokens `[t, ..., t+T-1]`
- target `Y` contains the next tokens `[t+1, ..., t+T]`

The common bug here is an off-by-one mistake in the target shift.

In [ ]:
def get_batch(data: np.ndarray, batch_size: int, T: int, seed: int = 0):
    rng = np.random.default_rng(seed)
    n = data.size
    starts = rng.integers(0, n - (T + 1), size=(batch_size,))
    X = np.stack([data[s:s+T] for s in starts], axis=0)
    Y = np.stack([data[s+1:s+T+1] for s in starts], axis=0)
    return X, Y

B, T = 8, 32
Xb, Yb = get_batch(data, B, T, seed=0)
print('Xb shape', Xb.shape, 'Yb shape', Yb.shape)


In [ ]:
# Checks
check('batch_shapes', Xb.shape == (B, T) and Yb.shape == (B, T))
check('shifted', np.all(Xb[:, 1:] == Yb[:, :-1]))
check('ids_in_range', int(Xb.min()) >= 0 and int(Xb.max()) < V and int(Yb.min()) >= 0 and int(Yb.max()) < V)


**Why this works.** Teacher forcing trains the model to predict the next token at every position. The target is exactly the input shifted left by one step.

## Task 4: Section 2A — Softmax, log-softmax, and causal mask
*Section: Model primitives*

## Section 2A — Model primitives

Implement three low-level utilities used by the LM:
- `softmax(...)`
- `log_softmax(...)`
- `causal_mask(T)`

Two details matter here:
- subtract the row max before exponentiating for numerical stability
- mask future positions **before** softmax so attention cannot see later tokens

In [ ]:
def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    m = np.max(x, axis=axis, keepdims=True)
    z = x - m
    e = np.exp(z)
    return e / np.sum(e, axis=axis, keepdims=True)

def log_softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    m = np.max(x, axis=axis, keepdims=True)
    z = x - m
    return z - np.log(np.sum(np.exp(z), axis=axis, keepdims=True))

def causal_mask(T: int) -> np.ndarray:
    return np.tril(np.ones((T, T), dtype=np.int64))[None, :, :]


In [ ]:
# Checks
sm = softmax(np.array([[0.0, 1.0, 2.0]], dtype=np.float64), axis=1)
ls = log_softmax(np.array([[0.0, 1.0, 2.0]], dtype=np.float64), axis=1)
m = causal_mask(4)
check('softmax_rowsum', float(np.max(np.abs(sm.sum(axis=1) - 1.0))) < 1e-8)
check('log_softmax_normalized', float(np.max(np.abs(np.exp(ls).sum(axis=1) - 1.0))) < 1e-8)
check('mask_is_lower_triangular', m.shape == (1, 4, 4) and np.array_equal(m[0], np.tril(np.ones((4, 4), dtype=np.int64))))


**Why this works.** These utilities are the numerical core of the notebook. `log_softmax` is used for stable cross-entropy, and the causal mask enforces autoregressive decoding by blocking future attention.

## Task 5: Section 2B — Initialize parameters and run the forward pass
*Section: Model forward*

## Section 2B — Model forward

Build a minimal **single-head causal self-attention LM**.

Forward pipeline:
1. `X -> H = E[X]`
2. `Q = H Wq`, `K = H Wk`, `V = H Wv`
3. `scores = QK^T / sqrt(D)`
4. apply causal mask
5. `A = softmax(scores)`
6. `O = A @ V`
7. `logits = O @ Wo + bo`

In [ ]:
$f

In [ ]:
# Checks
upper = np.triu_indices(T, k=1)
check('logits_shape', logits.shape == (B, T, V))
check('attn_rowsum', float(np.max(np.abs(cache['A'].sum(axis=-1) - 1.0))) < 1e-8)
check('future_attention_zero', float(np.max(np.abs(cache['A'][:, upper[0], upper[1]]))) < 1e-8)


**Why this works.** This is the smallest useful decoder-style LM: embeddings, one causal attention head, and a linear projection back to vocabulary logits.

## Task 6: Section 3 — Next-token cross-entropy
*Section: Loss*

## Section 3 — Loss

Implement average **next-token cross-entropy** over all `B*T` positions.

The clean pattern is:
- flatten `(B, T, V)` into `(B*T, V)`
- flatten targets `(B, T)` into `(B*T,)`
- use `log_softmax`
- compute both loss and `dlogits`

In [ ]:
def xent_loss_and_grad_logits(logits: np.ndarray, Y: np.ndarray):
    B, T, V = logits.shape
    flat_logits = logits.reshape(B * T, V)
    flat_y = Y.reshape(B * T)
    ls = log_softmax(flat_logits, axis=1)
    loss = -np.mean(ls[np.arange(B * T), flat_y])
    P = np.exp(ls)
    d = P
    d[np.arange(B * T), flat_y] -= 1.0
    d /= (B * T)
    return float(loss), d.reshape(B, T, V)

loss, dlogits = xent_loss_and_grad_logits(logits, Yb)
print('loss', loss)


In [ ]:
# Checks
flat_grad = dlogits.reshape(-1, dlogits.shape[-1])
check('dlogits_shape', dlogits.shape == logits.shape)
check('loss_finite', np.isfinite(loss) and loss > 0.0)
check('grad_rowsum_zero', float(np.max(np.abs(flat_grad.sum(axis=1)))) < 1e-8)


**Why this works.** Cross-entropy is computed over every time step in every sequence. The gradient with respect to logits is the standard `softmax(logits) - one_hot(target)` pattern, averaged over all positions.

## Task 7: Section 4 — Manual backprop through attention
*Section: Backprop*

## Section 4 — Backprop

Backpropagate through the full tiny LM:
- output projection `O @ Wo + bo`
- attention output `A @ V`
- row-wise softmax over attention scores
- score matrix `QK^T / sqrt(D)`
- projections `Wq, Wk, Wv`
- embedding lookup `E[X]`

This is the core mechanics section of the lab.

In [ ]:
$10

In [ ]:
# Checks
check('grad_keys', set(grads.keys()) == set(params.keys()))
check('grad_shapes', all(grads[k].shape == params[k].shape for k in params.keys()))
check('grad_finite', all(np.isfinite(grads[k]).all() for k in grads.keys()))


**Why this works.** The key Jacobian-vector product is the row-wise softmax backward rule: `dScores = A * (dA - sum(dA * A))`. After that, the rest is standard matrix backprop plus a scatter-add into the embedding table.

## Task 8: Section 5 — SGD training loop
*Section: Train*

## Section 5 — Train

Train the tiny LM for a small number of steps and verify that the loss decreases.

This is not about reaching great text quality. The goal is to confirm that:
- forward is wired correctly
- loss is computed correctly
- gradients push parameters in a useful direction

In [ ]:
def sgd_step(params: dict, grads: dict, lr: float):
    for k in params.keys():
        params[k] -= lr * grads[k]

def train_steps(params: dict, data: np.ndarray, steps: int = 150, batch_size: int = 16, T: int = 64, lr: float = 0.4, seed: int = 0):
    losses = []
    for t in range(steps):
        Xb, Yb = get_batch(data, batch_size, T, seed=seed + t)
        logits, cache = forward(params, Xb)
        loss, dlogits = xent_loss_and_grad_logits(logits, Yb)
        grads = backward(params, cache, dlogits)
        sgd_step(params, grads, lr=lr)
        losses.append(loss)
        if (t + 1) % 50 == 0:
            print('step', t + 1, 'loss', round(loss, 4))
    return losses

params2 = init_params(V, D, seed=1)
losses = train_steps(params2, data, steps=150, batch_size=16, T=64, lr=0.4, seed=0)
print('start/end loss', losses[0], losses[-1])


In [ ]:
# Checks
check('loss_history_nonempty', len(losses) == 150 and np.isfinite(np.array(losses)).all())
check('loss_down', losses[-1] < losses[0])


**Why this works.** A simple SGD loop is enough for this toy lab. If the loss reliably decreases, the forward pass, loss, and backprop are consistent enough to train.

## Task 9: Section 6 — Autoregressive sampling
*Section: Sampling*

## Section 6 — Sampling

Generate text one token at a time.

At each step:
1. take the recent context
2. run the model
3. read the last-step logits
4. convert to probabilities with temperature
5. sample the next token
6. append it to the context

In [ ]:
def sample(params: dict, start: str, n_new: int = 200, T: int = 64, temperature: float = 1.0, seed: int = 0) -> str:
    rng = np.random.default_rng(seed)
    ctx = list(encode(start))
    for _ in range(n_new):
        x = np.array(ctx[-T:], dtype=np.int64)[None, :]
        logits, _ = forward(params, x)
        last = logits[0, -1] / max(1e-6, float(temperature))
        p = softmax(last, axis=0)
        nxt = int(rng.choice(np.arange(V), p=p))
        ctx.append(nxt)
    return decode(np.array(ctx, dtype=np.int64))

out = sample(params2, start='the ', n_new=120, T=64, temperature=0.9, seed=0)
print(out)


In [ ]:
# Checks
check('sample_starts_with_prompt', out.startswith('the '))
check('sample_grows', len(out) > len('the '))


**Why this works.** Sampling turns the trained LM into a generator. The model repeatedly predicts a distribution over the next character and feeds its own output back as the next input context.